### set-up the project

In [287]:
import os
os.chdir(r"C:/Users/RHITI/Yasser_viz")

### Structure du Projet

In [288]:
import subprocess
subprocess.run("tree /F /A > Structure.txt", shell=True); print(open("Structure.txt").read())

Structure du dossier pour le volume OS
Le numéro de série du volume est E063-13A3
C:.
|   requirements.txt
|   Structure.txt
|   
+---venv
|   |   pyvenv.cfg
|   |   
|   +---etc
|   |   \---jupyter
|   |       \---nbconfig
|   |           \---notebook.d
|   |                   dash.json
|   |                   jupyterlab-plotly.json
|   |                   pydeck.json
|   |                   
|   +---Include
|   +---Lib
|   |   \---site-packages
|   |       |   .DS_Store
|   |       |   distutils-precedence.pth
|   |       |   nest_asyncio.py
|   |       |   numpy-2.0.2-cp39-cp39-win_amd64.whl
|   |       |   pylab.py
|   |       |   pythoncom.py
|   |       |   pywin32.pth
|   |       |   pywin32.version.txt
|   |       |   retrying.py
|   |       |   scipy-1.13.1-cp39-cp39-win_amd64.whl
|   |       |   six.py
|   |       |   threadpoolctl.py
|   |       |   typing_extensions.py
|   |       |   
|   |       +---adodbapi
|   |       |   |   adodbapi.py
|   |       |   |   ado_consts.p

Création d'un environnement virtuel pour ce projet afin d’éviter les conflits de dépendances :

1) VSCode terminal: Cliquer sur "Terminal" dans la bar

Vérifiez les fichiers existants : ls

2) Créez un environnement virtuel portant le nom venv : python -m venv venv

Vérifiez la création du dossier venv : ls

La création de l'environnement ne se fait qu'une seule fois.\
 Il ne reste qu'activer l'environnement chaque fois qu'on veut executer le projet:

3) Activez l'environnement : .\venv\Scripts\activate

...

4) Désactivez l'environnement après usage : deactivate

### Installation des bibliothèques Python nécessaire

In [289]:
#! pip install pandas numpy

In [290]:
#! pip install folium bokeh

In [291]:
#! pip install scikit-learn plotly geopandas seaborn statsmodels

In [292]:
#! pip install pandoc python-docx streamlit

Pour enregistrer les dépendances :

In [293]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


Data importation

In [294]:
os.chdir(r'C:/Users/RHITI/Yasser_viz/Yasser_viz/projet_agricole/data')

import pandas as pd
monit_cultures = pd.read_csv('monitoring_cultures.csv')
monit_cultures.head()

,date,parcelle_id,latitude,longitude,culture,ndvi,lai,stress_hydrique,biomasse_estimee
0,2020-01-01,P001,33.85339,-5.515999,sol_nu,0.181,0.15,0.0,0.0
1,2020-01-02,P001,33.85339,-5.515999,sol_nu,0.190,0.26,0.0,0.0
2,2020-01-03,P001,33.85339,-5.515999,sol_nu,0.164,0.21,0.0,0.0
3,2020-01-04,P001,33.85339,-5.515999,sol_nu,0.178,0.20,0.0,0.0
4,2020-01-05,P001,33.85339,-5.515999,sol_nu,0.140,0.18,0.0,0.0


In [295]:
import pandas as pd
met_detail = pd.read_csv('meteo_detaillee.csv')
met_detail.head()

,date,temperature,humidite,precipitation,rayonnement_solaire,vitesse_vent,direction_vent
0,2020-01-01 00:00:00,8.14,93.5,0.0,0.0,4.7,256.9
1,2020-01-01 01:00:00,7.40,95.0,0.0,0.0,5.0,251.7
2,2020-01-01 02:00:00,5.54,95.0,0.0,0.0,3.8,169.2
3,2020-01-01 03:00:00,3.88,95.0,0.0,0.0,3.4,137.0
4,2020-01-01 04:00:00,3.26,95.0,0.0,0.0,3.7,207.3


In [296]:
import pandas as pd
sols = pd.read_csv('sols.csv')
sols.head()

,parcelle_id,latitude,longitude,type_sol,surface_ha,capacite_retention_eau,ph,matiere_organique,azote,phosphore,potassium
0,P001,33.853390,-5.515999,argileux,5.07,0.89,7.9,3.62,0.254,46.0,255.8
1,P002,33.851719,-5.545701,argileux,19.72,0.81,7.5,4.03,0.243,48.6,299.6
2,P003,33.927878,-5.530586,sablo-limoneux,12.26,0.42,6.8,2.88,0.188,32.0,164.9
3,P004,33.854388,-5.585937,argilo-limoneux,12.70,0.74,6.9,3.92,0.184,59.6,252.3
4,P005,33.855558,-5.514110,argileux,17.32,0.81,7.8,3.70,0.300,41.1,294.0


In [297]:
import pandas as pd
hist_rendements = pd.read_csv('historique_rendements.csv')
hist_rendements.head()

,parcelle_id,date,culture,rendement_estime,rendement_final,progression
0,P001,2020-01-31,Ble,0.00,NaN,0.0
1,P001,2020-02-29,Ble,0.83,NaN,12.1
2,P001,2020-03-31,Ble,1.61,NaN,25.0
3,P001,2020-04-30,Mais,0.00,NaN,0.0
4,P001,2020-05-31,Mais,1.60,NaN,17.2


# Gestion et Analyse des Données

La Classe *AgriculturalDataManager* \
Cette classe centrale servira de point d’entrée pour toutes les opérations de données dans notre système.

In [298]:
pip install scikit-learn

In [299]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [300]:
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource

class AgriculturalDataManager:
    def __init__(self):
        self.monitoring_data = None
        self.weather_data = None
        self.soil_data = None
        self.yield_history = None
        self.features = None
    
    def load_data(self):
        # Simulated data loading, replace with actual paths or data sources
        self.monitoring_data = pd.DataFrame({
            'date': pd.date_range('2024-01-01', periods=5, freq='D'),
            'parcelle_id': ['P026']*5,
            'culture': ['Mais']*5,
            'ndvi': [0.442, 0.313, 0.454, 0.419, 0.462],
            'lai': [0.7, 0.71, 0.57, 1.23, 0.69],
            'stress_hydrique': [0.101, 0.055, 0.082, 0.049, 0.052],
            'biomasse_estimee': [3.09, 2.22, 2.60, 5.14, 3.20],
            'latitude': [33.866186]*5,
            'longitude': [-5.559177]*5
        })

        self.weather_data = pd.DataFrame({
            'date': pd.date_range('2024-01-01', periods=5, freq='D'),
            'temperature': [11.29, 11.60, 12.32, 11.49, 10.59],
            'precipitation': [0.0, 0.1, 0.0, 0.0, 0.0],
            'humidite': [0.8, 0.79, 0.78, 0.77, 0.76]
        })

        self.soil_data = pd.DataFrame({
            'parcelle_id': ['P026']*5,
            'type_sol': ['argilo-limoneux']*5,
            'surface_ha': [11.01]*5,
            'capacite_retention_eau': [0.79]*5,
            'ph': [7.2]*5
        })

        self.yield_history = pd.DataFrame({
            'parcelle_id': ['P026', 'P026', 'P026', 'P026', 'P026'],
            'annee': [2020, 2021, 2022, 2023, 2024],
            'culture': ['Mais', 'Mais', 'Tournesol', 'Ble', 'Mais'],
            'yield': [12.3, 13.95, 2.47, 6.75, 11.75]
        })
        
    def prepare_features(self):
        self.monitoring_data['date'] = pd.to_datetime(self.monitoring_data['date'])
        self.weather_data['date'] = pd.to_datetime(self.weather_data['date'])

        # Merging dataframes on 'date' and 'parcelle_id'
        self.features = pd.merge(self.monitoring_data, self.weather_data, on='date', how='left')
        self.features = pd.merge(self.features, self.soil_data, on='parcelle_id', how='left')
        self.features = pd.merge(self.features, self.yield_history, on=['parcelle_id', 'culture'], how='left')

        # Extracting month
        self.features['month'] = self.features['date'].dt.month
        
        # Ensuring no missing data
        for name, df in {'monitoring_data': self.monitoring_data, 
                         'weather_data': self.weather_data, 
                         'soil_data': self.soil_data, 
                         'yield_history': self.yield_history}.items():
            print(f"Pas de valeurs manquantes dans {name}" if df.isnull().sum().sum() == 0 else f"Des valeurs manquantes dans {name}")

        print("Features Data:\n", self.features.head())
        return self.features
    
    def plot_features(self):
        source = ColumnDataSource(self.features)
        
        # Bokeh plot example
        p = figure(x_axis_type='datetime', title="NDVI and Yield Over Time")
        p.line(x='date', y='ndvi', source=source, legend_label="NDVI", line_width=2, color="green")
        p.line(x='date', y='yield', source=source, legend_label="Yield", line_width=2, color="blue")
        p.legend.location = "top_left"
        
        show(p)


# Usage
data_manager = AgriculturalDataManager()
data_manager.load_data()
features = data_manager.prepare_features()
data_manager.plot_features()


Pas de valeurs manquantes dans monitoring_data
Pas de valeurs manquantes dans weather_data
Pas de valeurs manquantes dans soil_data
Pas de valeurs manquantes dans yield_history
Features Data:
         date parcelle_id culture   ndvi  lai  stress_hydrique  \
0 2024-01-01        P026    Mais  0.442  0.7            0.101   
1 2024-01-01        P026    Mais  0.442  0.7            0.101   
2 2024-01-01        P026    Mais  0.442  0.7            0.101   
3 2024-01-01        P026    Mais  0.442  0.7            0.101   
4 2024-01-01        P026    Mais  0.442  0.7            0.101   

   biomasse_estimee   latitude  longitude  temperature  precipitation  \
0              3.09  33.866186  -5.559177        11.29            0.0   
1              3.09  33.866186  -5.559177        11.29            0.0   
2              3.09  33.866186  -5.559177        11.29            0.0   
3              3.09  33.866186  -5.559177        11.29            0.0   
4              3.09  33.866186  -5.559177        11

# Utilisation de la classe
data_manager = AgriculturalDataManager()
data_manager.load_data()

features = data_manager.prepare_features()
    
parcelle_id = 'P001'
history, trend = data_manager.get_temporal_patterns(parcelle_id)

risk_metrics = data_manager.calculate_risk_metrics(features)

print(f"Tendance de rendement : {trend['pente']:.2f} tonnes/ha/jour")
print(f"Variation moyenne : {trend['variation_moyenne']*100:.1f}%")

print("\nMétriques de risque (5 premières lignes) :")
print(risk_metrics[['parcelle_id', 'date', 'ndvi', 'stress_hydrique', 'risk_score']].head())

In [301]:
pip install bokeh

Note: you may need to restart the kernel to use updated packages.


In [302]:
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, Select, DateRangeSlider, HoverTool, ColorBar, LinearColorMapper
from bokeh.plotting import figure, show
from bokeh.palettes import RdYlBu11 as palette
from bokeh.transform import linear_cmap
import pandas as pd
import numpy as np

class AgriculturalDashboard:

    def __init__(self, data_manager):
        """
        Initialise le tableau de bord avec le gestionnaire de données.
        Le gestionnaire de données (data_manager) doit avoir chargé :
        - Les données de monitoring actuelles
        - L'historique des rendements
        - Les données météorologiques
        - Les caractéristiques des sols
        """
        self.data_manager = data_manager
        self.source = None
        self.hist_source = None
        self.selected_parcelle = None
        self.create_data_sources()

    def create_data_sources(self):
        """
        Prépare les sources de données pour Bokeh en intégrant
        les données actuelles et historiques.
        """
        monitoring_data = self.data_manager.monitoring_data
        yield_history = self.data_manager.yield_history

        self.source = ColumnDataSource(monitoring_data)
        self.hist_source = ColumnDataSource(yield_history)

    def create_yield_history_plot(self):
        """
        Crée un graphique montrant l'évolution historique des rendements
        avec des annotations pour les événements importants.
        """
        p = figure(title='Historique des Rendements par Parcelle',
                   x_axis_type='datetime',
                   height=400, width=800)
        p.line('annee', 'rendement', source=self.hist_source, line_width=2, color='green')
        p.circle('annee', 'rendement', source=self.hist_source, size=8, color='blue')

        hover = HoverTool(tooltips=[('Année', '@annee{%Y}'), ('Rendement', '@rendement')], 
                          formatters={'@annee': 'datetime'})
        p.add_tools(hover)
        p.xaxis.axis_label = "Année"
        p.yaxis.axis_label = "Rendement (t/ha)"

        return p

    def create_ndvi_temporal_plot(self):
        """
        Crée un graphique montrant l’évolution du NDVI avec
        des seuils de référence basés sur l’historique.
        """
        p = figure(title='Évolution du NDVI et Seuils Historiques',
                   x_axis_type='datetime',
                   height=400, width=800)
        p.line('date', 'ndvi', source=self.source, line_width=2, color='blue', legend_label="NDVI")
        p.line('date', 'ndvi_threshold', source=self.source, line_width=2, color='red', legend_label="Seuil")

        hover = HoverTool(tooltips=[('Date', '@date{%F}'), ('NDVI', '@ndvi')],
                          formatters={'@date': 'datetime'})
        p.add_tools(hover)
        p.xaxis.axis_label = "Date"
        p.yaxis.axis_label = "NDVI"

        return p

    def create_stress_matrix(self):
        """
        Crée une matrice de stress combinant stress hydrique
        et conditions météorologiques.
        """
        p = figure(title='Matrice de Stress',
                   x_range=[0, 1], y_range=[0, 1],
                   height=400, width=800)

        # On vérifie si les données sont correctement préparées
        try:
            stress_data = self.data_manager.prepare_features()
            if stress_data is None:
                raise ValueError("Données de stress non disponibles.")
            
            self.source.data = stress_data

            # Si les données sont prêtes, on applique la carte des couleurs
            mapper = linear_cmap(field_name='stress_level', palette=palette, low=0, high=1)
            p.circle(x='stress_hydrique', y='stress_meteo', size=10, source=self.source, color=mapper)

            color_bar = ColorBar(color_mapper=mapper['transform'], width=8, location=(0, 0))
            p.add_layout(color_bar, 'right')

            p.xaxis.axis_label = "Stress Hydrique"
            p.yaxis.axis_label = "Stress Météorologique"

        except Exception as e:
            print(f"Erreur dans la préparation des données de stress: {e}")
            p = figure(title="Erreur dans la préparation des données", height=400, width=800)
            p.text(x=0.5, y=0.5, text=["Erreur dans les données de stress"], text_align="center", text_font_size="16pt")
        
        return p

    def create_yield_prediction_plot(self):
        """
        Crée un graphique de prédiction des rendements
        basé sur les données historiques et actuelles.
        """
        p = figure(title='Prédiction des Rendements',
                   x_axis_type='datetime',
                   height=400, width=800)

        p.line('date', 'predicted_yield', source=self.source, line_width=2, color='orange', legend_label="Prédiction")
        p.line('date', 'actual_yield', source=self.source, line_width=2, color='green', legend_label="Réalité")

        hover = HoverTool(tooltips=[('Date', '@date{%F}'), ('Prédiction', '@predicted_yield'), ('Réel', '@actual_yield')], 
                          formatters={'@date': 'datetime'})
        p.add_tools(hover)
        p.xaxis.axis_label = "Date"
        p.yaxis.axis_label = "Rendement (t/ha)"

        return p

    def create_layout(self):
        """
        Organise tous les graphiques dans une mise en page cohérente.
        """
        yield_plot = self.create_yield_history_plot()
        ndvi_plot = self.create_ndvi_temporal_plot()
        stress_plot = self.create_stress_matrix()
        prediction_plot = self.create_yield_prediction_plot()

        layout = column(yield_plot, ndvi_plot, stress_plot, prediction_plot)
        return layout

    def update_plots(self, attr, old, new):
        """
        Met à jour tous les graphiques quand une nouvelle parcelle est sélectionnée.
        """
        self.selected_parcelle = new
        filtered_data = self.data_manager.monitoring_data[
            self.data_manager.monitoring_data['parcelle_id'] == new
        ]
        self.source.data = ColumnDataSource(filtered_data).data

# Exemple d’utilisation (avec un data_manager fictif)
dashboard = AgriculturalDashboard(data_manager)
layout = dashboard.create_layout()
show(layout)


Pas de valeurs manquantes dans monitoring_data
Pas de valeurs manquantes dans weather_data
Pas de valeurs manquantes dans soil_data
Pas de valeurs manquantes dans yield_history
Features Data:
         date parcelle_id culture   ndvi  lai  stress_hydrique  \
0 2024-01-01        P026    Mais  0.442  0.7            0.101   
1 2024-01-01        P026    Mais  0.442  0.7            0.101   
2 2024-01-01        P026    Mais  0.442  0.7            0.101   
3 2024-01-01        P026    Mais  0.442  0.7            0.101   
4 2024-01-01        P026    Mais  0.442  0.7            0.101   

   biomasse_estimee   latitude  longitude  temperature  precipitation  \
0              3.09  33.866186  -5.559177        11.29            0.0   
1              3.09  33.866186  -5.559177        11.29            0.0   
2              3.09  33.866186  -5.559177        11.29            0.0   
3              3.09  33.866186  -5.559177        11.29            0.0   
4              3.09  33.866186  -5.559177        11

ERROR:bokeh.core.validation.check:E-1001 (BAD_COLUMN_NAME): Glyph refers to nonexistent column name. This could either be due to a misspelling or typo, or due to an expected column being missing. : y='rendement' [no close matches] {renderer: GlyphRenderer(id='p4070', ...)}
ERROR:bokeh.core.validation.check:E-1001 (BAD_COLUMN_NAME): Glyph refers to nonexistent column name. This could either be due to a misspelling or typo, or due to an expected column being missing. : fill_color='stress_level' [no close matches], hatch_color='stress_level' [no close matches], line_color='stress_level' [no close matches], y='stress_meteo' [no close matches] {renderer: GlyphRenderer(id='p4187', ...)}
ERROR:bokeh.core.validation.check:E-1001 (BAD_COLUMN_NAME): Glyph refers to nonexistent column name. This could either be due to a misspelling or typo, or due to an expected column being missing. : y='actual_yield' [no close matches] {renderer: GlyphRenderer(id='p4256', ...)}
ERROR:bokeh.core.validation.check

In [303]:
%matplotlib inline
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [310]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
import warnings
import os
warnings.filterwarnings('ignore')

# Définir le répertoire de travail
os.chdir(r'C:/Users/RHITI/Yasser_viz/Yasser_viz/projet_agricole/data')

class AgriculturalDataManager:
    def __init__(self):
        """Initialise le gestionnaire de données agricoles"""
        self.monitoring_data = None
        self.weather_data = None
        self.soil_data = None
        self.yield_history = None
        self.scaler = StandardScaler()

    def load_data(self):
        """ 
        Charge l'ensemble des données nécessaires au système
        Effectue les conversions de types et les indexations temporelles 
        """
        try:
            self.monitoring_data = pd.read_csv('monitoring_cultures.csv', parse_dates=['date'])
            self.weather_data = pd.read_csv('meteo_detaillee.csv', parse_dates=['date'])
            self.soil_data = pd.read_csv('sols.csv')
            self.yield_history = pd.read_csv('historique_rendements.csv')
        except FileNotFoundError as e:
            raise FileNotFoundError(f"Erreur lors du chargement des données : {e}")

        self._setup_temporal_indices()
        self._verify_temporal_consistency()
        self._check_missing_values()
        self._check_unit_consistency()

    def _setup_temporal_indices(self):
        """Configure les index temporels pour les différentes séries de données et vérifie leur cohérence"""
        self.monitoring_data.set_index('date', inplace=True)
        self.monitoring_data.sort_index(inplace=True)
        self.weather_data.set_index('date', inplace=True)
        self.weather_data.sort_index(inplace=True)
        self.yield_history['mois'] = pd.to_datetime(self.yield_history['date']).dt.month

    def _verify_temporal_consistency(self):
        """Vérifie la cohérence des périodes temporelles entre les différents jeux de données"""
        monitoring_range = self.monitoring_data.index
        weather_range = self.weather_data.index
        if not (weather_range.min() <= monitoring_range.min() and weather_range.max() >= monitoring_range.max()):
            warnings.warn("Les données de monitoring dépassent la plage des données météo")

    def _check_missing_values(self):
        for df_name, df in [("monitoring_data", self.monitoring_data),
                            ("weather_data", self.weather_data),
                            ("soil_data", self.soil_data),
                            ("yield_history", self.yield_history)]:
            missing = df.isnull().sum()
            if missing.any():
                print(f"Valeurs manquantes dans {df_name}:")
                print(missing[missing > 0])
            else:
                print(f"Pas de valeurs manquantes dans {df_name}")

    def _check_unit_consistency(self):
        """Vérifie la cohérence des unités de rendement dans yield_history"""
        if 'rendement_final' not in self.yield_history.columns:
            print("La colonne 'rendement_final' est manquante dans 'yield_history'. Voici les colonnes disponibles :")
            print(self.yield_history.columns)
        else:
            yield_units = self.yield_history.groupby('mois')['rendement_final'].mean()
            if yield_units.std() / yield_units.mean() > 0.5:  # Seuil arbitraire de 50% de variation
                warnings.warn("Possible incohérence dans les unités de rendement entre les mois")

    def prepare_features(self):
        """Prépare les caractéristiques pour l'analyse en fusionnant les différentes sources de données"""
        merged_data = pd.merge_asof(self.monitoring_data.reset_index(),
                                    self.weather_data.reset_index(),
                                    on='date',
                                    direction='nearest')
        merged_data = pd.merge(merged_data, self.soil_data, on='parcelle_id')
        return self._enrich_with_yield_history(merged_data)

    def _enrich_with_yield_history(self, data):
        """Enrichit les données actuelles avec les informations historiques des rendements"""
        avg_yields = self.yield_history.groupby(['parcelle_id', 'culture'])['rendement_final'].mean().reset_index()
        avg_yields.columns = ['parcelle_id', 'culture', 'rendement_moyen_historique']
        return pd.merge(data, avg_yields, on=['parcelle_id', 'culture'], how='left')

    def get_temporal_patterns(self, parcelle_id):
        """Analyse les patterns temporels pour une parcelle donnée"""
        parcel_data = self.monitoring_data[self.monitoring_data['parcelle_id'] == parcelle_id]
        if len(parcel_data) < 2:
            raise ValueError("Données insuffisantes pour le calcul de tendance")

        rolling_metrics = parcel_data[['ndvi', 'lai', 'stress_hydrique', 'biomasse_estimee']].rolling(window=7).mean()
        time_index = (parcel_data.index - parcel_data.index[0]).days
        time_weights = np.exp(-(time_index.max() - time_index) / 365)  # Donne plus de poids aux données récentes
        trend = np.polyfit(time_index, parcel_data['biomasse_estimee'], 1, w=time_weights)
        slope = trend[0]
        variation = parcel_data['biomasse_estimee'].pct_change().mean()

        return rolling_metrics, {'pente': slope, 'variation_moyenne': variation}

    def calculate_risk_metrics(self, data, stress_weight=0.6, ndvi_weight=0.2, weather_weight=0.2):
        """Calcule les métriques de risque basées sur les conditions actuelles et l'historique"""
        data['ndvi_ecart'] = data['ndvi'] - data.groupby('parcelle_id')['ndvi'].transform('mean')
        data['extreme_weather'] = ((data['temperature'] > data['temperature'].quantile(0.95)) | 
                                   (data['precipitation'] > data['precipitation'].quantile(0.95))).astype(int)
        data['risk_score'] = (data['stress_hydrique'] * stress_weight + 
                              data['ndvi_ecart'].abs() * ndvi_weight +
                              data['extreme_weather'] * weather_weight) * 100
        return data

# Utilisation de la classe
data_manager = AgriculturalDataManager()
data_manager.load_data()

features = data_manager.prepare_features()

parcelle_id = 'P001'
history, trend = data_manager.get_temporal_patterns(parcelle_id)

risk_metrics = data_manager.calculate_risk_metrics(features)

print(f"Tendance de rendement : {trend['pente']:.2f} tonnes/ha/jour")
print(f"Variation moyenne : {trend['variation_moyenne']*100:.1f}%")

# Affichage des 5 premières lignes des métriques de risque
print("\nMétriques de risque (5 premières lignes) :")
print(risk_metrics[['parcelle_id', 'date', 'ndvi', 'stress_hydrique', 'extreme_weather', 'risk_score']].head())


Pas de valeurs manquantes dans monitoring_data
Pas de valeurs manquantes dans weather_data
Pas de valeurs manquantes dans soil_data
Valeurs manquantes dans yield_history:
rendement_final    2494
dtype: int64
Tendance de rendement : -0.00 tonnes/ha/jour
Variation moyenne : inf%

Métriques de risque (5 premières lignes) :
  parcelle_id       date   ndvi  stress_hydrique  extreme_weather  risk_score
0        P001 2020-01-01  0.181              0.0                0    6.472206
1        P028 2020-01-01  0.108              0.0                0    7.145068
2        P011 2020-01-01  0.103              0.0                0    7.687849
3        P046 2020-01-01  0.177              0.0                0    5.833695
4        P027 2020-01-01  0.131              0.0                0    7.056749


In [663]:
#pip install pandas numpy scikit-learn

In [690]:
import matplotlib.pyplot as plt
# Step 2: Import Necessary Libraries
import folium
from folium import plugins
from branca.colormap import LinearColormap
import numpy as np
import pandas as pd

# Step 3: Define the Data Manager (Mock)
class DataManager:
    def __init__(self):
        self.data = {
            'locations': [[46.2276, 2.2137], [47.2138, 1.5375], [45.7640, 4.8357]],  # Latitude, Longitude
            'yields': [5, 8, 12],  # Yield values
             'history': {
            (46.2276, 2.2137): {'years': [2020, 2021, 2022], 'yields': [4, 6, 5], 'crops': ['wheat', 'barley', 'wheat']},
            (47.2138, 1.5375): {'years': [2020, 2021, 2022], 'yields': [7, 9, 8], 'crops': ['corn', 'soybean', 'corn']},
            (45.7640, 4.8357): {'years': [2020, 2021, 2022], 'yields': [11, 13, 12], 'crops': ['sunflower', 'wheat', 'sunflower']},
        },
        'ndvi_data': pd.DataFrame({
            'latitude': [46.2276, 47.2138, 45.7640],
            'longitude': [2.2137, 1.5375, 4.8357],
            'ndvi': [0.7, 0.5, 0.8],
            'date': ['2024-03-01', '2024-03-01', '2024-03-01']  # Example date
        }),
        'risk_data' : {
            'locations': [[46.2276, 2.2137], [47.2138, 1.5375], [45.7640, 4.8357]],  # Latitude, Longitude
            'risk_levels': [2, 1, 3] # Risk level
        },
         'crop_data': pd.DataFrame({
            'latitude': [46.2276, 47.2138, 45.7640],
            'longitude': [2.2137, 1.5375, 4.8357],
            'crop_name': ['wheat', 'corn', 'sunflower'],
            'growth_stage': ['vegetative', 'reproductive', 'flowering'],
            'predicted_yield': [5.6, 8.9, 10.2],
            'area': [10, 12, 15]
        })

        }

    def get_yield_data(self):
        return self.data
    
    def get_history_data(self):
        return self.data['history']
    
    def get_ndvi_data(self):
        return self.data['ndvi_data']
    
    def get_risk_data(self):
         return self.data['risk_data']
    
    def get_crop_data(self):
        return self.data['crop_data']

data_manager = DataManager()

# Step 4: Define the AgriculturalMap Class
class AgriculturalMap:
    def __init__(self, data_manager):
        self.data_manager = data_manager
        self.map = None
        self.yield_colormap = LinearColormap(colors=['red', 'yellow', 'green'], vmin=0, vmax=12)
        self.risk_colormap = LinearColormap(colors=['green','yellow','red'], vmin=1, vmax=3)

    def create_base_map(self):
        """
        Crée la carte de base avec les couches appropriées
        """
        # Initialiser la carte Folium centrée sur les parcelles
        data = self.data_manager.get_yield_data()
        center_lat = sum([loc[0] for loc in data['locations']]) / len(data['locations'])
        center_lon = sum([loc[1] for loc in data['locations']]) / len(data['locations'])
        self.map = folium.Map(location=[center_lat, center_lon], zoom_start=6)

    def add_yield_history_layer(self):
        """
        Ajoute une couche visualisant l'historique des rendements
        """
        if self.map is None:
            self.create_base_map()
        
        history_data = self.data_manager.get_history_data()

        for location, history in history_data.items():
           
            mean_yield = np.mean(history['yields'])
            trend = self._calculate_yield_trend(history)
            popup_content = self._create_yield_popup(history, mean_yield, trend)
            
            folium.CircleMarker(
                    location=location,
                    radius=10,
                    color=self.yield_colormap(mean_yield),
                    fill=True,
                    fill_color=self.yield_colormap(mean_yield),
                    fill_opacity=0.7,
                    popup = popup_content
                ).add_to(self.map)

    def add_current_ndvi_layer(self):
       """
       Ajoute une couche de la situation NDVI actuelle
       """
       if self.map is None:
           self.create_base_map()
           
       ndvi_df = self.data_manager.get_ndvi_data()
       for index, row in ndvi_df.iterrows():
            popup_content = self._create_ndvi_popup(row)
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=10,
                color='blue',
                fill=True,
                fill_color='blue',
                fill_opacity=0.5,
                popup=popup_content
            ).add_to(self.map)


    def add_risk_heatmap(self):
         """
         Ajoute une carte de chaleur des zones à risque
         """
         if self.map is None:
             self.create_base_map()
         
         risk_data = self.data_manager.get_risk_data()
         for location, risk_level in zip(risk_data['locations'],risk_data['risk_levels']):
              folium.CircleMarker(
                location=location,
                radius=10,
                color=self.risk_colormap(risk_level),
                fill=True,
                fill_color=self.risk_colormap(risk_level),
                fill_opacity=0.7,
                popup=f"Risk level: {risk_level}"
            ).add_to(self.map)

    def add_crop_layer(self):
        """
        Ajoute une couche avec les informations sur les cultures
        """
        if self.map is None:
            self.create_base_map()

        crop_df = self.data_manager.get_crop_data()
        self._check_crop_data_columns(crop_df)


        for index, row in crop_df.iterrows():
            popup_content = self._create_crop_popup(row)
            folium.Marker(
                location=[row['latitude'], row['longitude']],
                popup = popup_content
            ).add_to(self.map)
            

    def _check_crop_data_columns(self, crop_df):
        """
        Vérifie la présence des colonnes nécessaires dans le DataFrame des cultures
        """
        colonnes_requises = ['latitude', 'longitude', 'crop_name', 'growth_stage', 'predicted_yield', 'area']
        colonnes_manquantes = [col for col in colonnes_requises if col not in crop_df.columns]
        if colonnes_manquantes:
            raise ValueError(f"Les colonnes suivantes sont manquantes dans le fichier de données des cultures: {', '.join(colonnes_manquantes)}")
        
    
    def _calculate_yield_trend(self, history):
            """
            Calcule la tendance des rendements pour une parcelle
            """
            if len(history['years']) < 2 :
                return 'No trend'
            
            years = np.array(history['years'])
            yields = np.array(history['yields'])
            
            slope = np.polyfit(years,yields,1)[0]
            
            if slope > 0:
                return 'Increasing'
            elif slope < 0 :
                return 'Decreasing'
            else:
                return 'Stable'

    def _create_yield_popup(self, history, mean_yield, trend):
            """
            Crée le contenu HTML du popup pour l'historique des rendements
            """
            formatted_crops = self._format_recent_crops(history)

            return f"""
                <div style="font-size: 12px;">
                    <b>Mean Yield:</b> {mean_yield:.2f} tonnes/ha<br>
                    <b>Yield Trend:</b> {trend}<br>
                    <b>Recent Crops:</b><br>
                     {formatted_crops}
                </div>
             """

    def _format_recent_crops(self, history):
        """
        Formate la liste des cultures récentes pour le popup
        """
        
        crops_html = "<ul>"
        for year, crop in zip(history['years'], history['crops']):
            crops_html += f"<li>{year}: {crop}</li>"
        crops_html += "</ul>"
        return crops_html

    def _create_ndvi_popup(self, row):
        """
        Crée le contenu HTML du popup pour les données NDVI actuelles
        """
        return f"""
            <div style="font-size: 12px;">
                <b>Date:</b> {row['date']}<br>
                <b>NDVI:</b> {row['ndvi']:.2f}
            </div>
        """
    
    def _create_crop_popup(self, row):
            """
            Crée le contenu HTML du popup pour les données des cultures
            """
            return f"""
                <div style="font-size: 12px;">
                    <b>Crop:</b> {row['crop_name']}<br>
                    <b>Growth Stage:</b> {row['growth_stage']}<br>
                    <b>Predicted Yield:</b> {row['predicted_yield']:.2f} tonnes/ha<br>
                    <b>Area:</b> {row['area']:.2f} ha
                </div>
            """


# Step 5: Instantiate the AgriculturalMap and Add Layers
agricultural_map = AgriculturalMap(data_manager)
agricultural_map.create_base_map()
agricultural_map.add_yield_history_layer()
agricultural_map.add_current_ndvi_layer()
agricultural_map.add_risk_heatmap()
agricultural_map.add_crop_layer()
agricultural_map.map

In [692]:
import streamlit as st
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource

def create_streamlit_dashboard():
    st.title("Tableau de Bord Agricole Intégré")
    data_manager = AgriculturalDataManager()
    data_manager.load_data()
    prepared_data = data_manager.prepare_features()
    # Afficher les visualisations Bokeh et Folium

In [666]:
#pip install streamlit-folium

In [667]:
 #pip install --force-reinstall --no-deps bokeh==2.4.3

In [6]:
import streamlit as st
from bokeh.plotting import figure
from bokeh.embed import components
import folium

class AgriculturalDashboard:
    def __init__(self):
        """Initialise les graphiques Bokeh."""
        self.plot = None

    def initialize_charts(self):
        """Créer un graphique Bokeh d'exemple."""
        self.plot = figure(title="Production Agricole", x_axis_label="Année", y_axis_label="Production (tonnes)")
        self.plot.line([2015, 2016, 2017, 2018, 2019, 2020], [100, 120, 130, 115, 140, 150], line_width=2)

    def update_charts(self, parcelle_id):
        """Met à jour les graphiques (exemple basique)."""
        if parcelle_id:
            self.plot.title.text = f"Production Agricole - Parcelle {parcelle_id}"

In [2]:
def initialize_visualizations(self):
    """
    Initialise toutes les composantes visuelles
    """
    self.bokeh_dashboard.initialize_charts()  # Exemple pour Bokeh
    self.map_view.initialize_map()            # Exemple pour Folium

In [3]:
def create_streamlit_dashboard(self):
    import streamlit as st
    from bokeh.embed import components
    from bokeh.resources import CDN

    st.title("Tableau de Bord Agricole Intégré")
    # Visualisation Bokeh
    script, div = components(self.bokeh_dashboard.plot)
    st.markdown(script + div, unsafe_allow_html=True)

    # Visualisation Folium
    map_html = self.map_view.get_map_html()  # Assurez-vous que cette méthode retourne l'HTML de Folium
    st.components.v1.html(map_html, height=600)

In [4]:
def update_visualizations(self, parcelle_id):
    """
    Met à jour toutes les visualisations pour une parcelle donnée
    """
    self.bokeh_dashboard.update_charts(parcelle_id)
    self.map_view.update_map(parcelle_id)

In [5]:
def setup_interactions(self):
    """
    Configure les interactions entre les composantes
    """
    # Exemple d'interaction Bokeh
    self.bokeh_dashboard.add_selection_callback(self.handle_parcelle_selection)

    # Exemple d'interaction Folium (par survol)
    self.map_view.add_hover_callback(self.handle_map_hover)

def handle_parcelle_selection(self, attr, old, new):
    self.update_visualizations(new)

def handle_map_hover(self, feature):
    st.sidebar.info(f"Survol : {feature['properties']['name']}")

In [7]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from datetime import datetime


class AgriculturalAnalyzer:
    def __init__(self, data_manager):
        """
        Initialize the analyzer with the data manager.

        This class uses historical and current data to generate relevant agronomic insights.
        """
        self.data_manager = data_manager
        self.model = RandomForestRegressor(n_estimators=100, random_state=42)

    def analyze_yield_factors(self, parcelle_id):
        """
        Analyze the factors influencing yields.

        This method combines historical data with current soil and climate characteristics
        to identify key performance factors.
        """
        historical_data = self.data_manager.get_history()
        if parcelle_id not in historical_data:
            raise ValueError("Parcelle ID not found in historical data.")
        
        data = historical_data[parcelle_id]
        yield_data = data['yields']
        years = data['years']

        # Simulating environmental factors for analysis
        weather_data = np.random.rand(len(years))  # Dummy weather data
        soil_quality = np.random.rand(len(years))  # Dummy soil quality data

        correlations = self._calculate_yield_correlations(yield_data, weather_data, soil_quality)
        limiting_factors = self._identify_limiting_factors(data, correlations)
        trends = self._analyze_performance_trend(data)

        return {
            "correlations": correlations,
            "limiting_factors": limiting_factors,
            "performance_trends": trends,
        }

    def _calculate_yield_correlations(self, yield_data, weather_data, soil_data):
        """
        Calculate correlations between yields and environmental factors.
        """
        weather_corr = pearsonr(yield_data, weather_data)[0]
        soil_corr = pearsonr(yield_data, soil_data)[0]
        return {"weather": weather_corr, "soil": soil_corr}

    def _identify_limiting_factors(self, parcelle_data, correlations):
        """
        Identify yield-limiting factors based on Liebig's Law of the Minimum.
        """
        limiting_factor = min(correlations, key=correlations.get)
        limiting_value = correlations[limiting_factor]
        return {limiting_factor: limiting_value}

    def _analyze_performance_trend(self, parcelle_data):
        """
        Analyze performance trends over time for the parcel.
        """
        years = parcelle_data['years']
        yields = parcelle_data['yields']
        trend = np.polyfit(years, yields, 1)[0]
        return {"trend_slope": trend, "trend_description": "Increasing" if trend > 0 else "Decreasing"}

    def _detect_yield_breakpoints(self, yield_series):
        """
        Detect significant changes in the yield time series.
        """
        diffs = np.diff(yield_series)
        breakpoints = np.where(np.abs(diffs) > np.std(diffs))[0]
        return {"breakpoints": breakpoints.tolist(), "break_values": yield_series[breakpoints].tolist()}

    def _analyze_yield_stability(self, yield_series):
        """
        Analyze the stability of yields over time.
        """
        stability_index = self._calculate_stability_index(yield_series)
        variability = np.std(yield_series)
        return {"stability_index": stability_index, "variability": variability}

    def _calculate_stability_index(self, yield_series):
        """
        Calculate a custom stability index.
        """
        trend = np.polyfit(range(len(yield_series)), yield_series, 1)[0]
        variability = np.std(yield_series)
        return trend / (1 + variability)

    def predict_future_yields(self, features):
        """
        Predict future yields using a trained machine learning model.
        """
        historical_data = self.data_manager.get_history()
        X = []
        y = []

        # Combine data for training (dummy features and labels for example)
        for data in historical_data.values():
            years = data['years']
            yields = data['yields']
            environmental_factors = np.random.rand(len(years), 2)  # Dummy environmental features
            X.extend(environmental_factors)
            y.extend(yields)

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        self.model.fit(X_train, y_train)

        return self.model.predict(features)